# 🌦️ Weather Data Analysis using Python, Pandas and NumPy
**Mini Project – 1st Year B.Tech**
**Submitted by:** Adarsh Tiwari

---
### What this project does
1. Downloads **real daily weather data** for a city (free API, no login needed)
2. **Cleans** the data using Pandas
3. **Analyses** it using NumPy and Pandas
4. Draws **charts** to understand temperature and rainfall patterns

> 💡 **How to run:** click **Runtime → Run all**, or press **Shift + Enter** on each cell from top to bottom.

## Step 1: Import the libraries
- **NumPy** – fast maths on numbers (mean, max, correlation, trend line)
- **Pandas** – tables of data (called *DataFrames*)
- **Matplotlib** – drawing charts
- **Requests** – downloading data from the internet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

print("Libraries imported successfully!")

## Step 2: Choose the city and years
Change `CITY` to **your own city** and run the notebook again to see its weather!

In [ ]:
CITY = "Delhi"          # <- change this to any city
START_YEAR = 2016       # first year of data
END_YEAR = 2025         # last year of data
USE_ONLINE_DATA = True  # True = real data, False = practice (sample) data

## Step 3: Get the weather data
Two functions are defined here:
- `get_online_data()` downloads real data from the free **Open-Meteo** website.
- `get_sample_data()` creates practice data with NumPy, so the project still works **without internet**.

If the download fails, the program automatically uses the sample data.

In [ ]:
def get_online_data(city, start_year, end_year):
    # 1) find the latitude and longitude of the city
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"
    geo = requests.get(geo_url, params={"name": city, "count": 1}, timeout=30).json()
    place = geo["results"][0]

    # 2) download daily weather for that location
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": place["latitude"],
        "longitude": place["longitude"],
        "start_date": f"{start_year}-01-01",
        "end_date": f"{end_year}-12-31",
        "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
        "timezone": "auto",
    }
    data = requests.get(url, params=params, timeout=120).json()["daily"]

    # 3) put it in a Pandas table with simple column names
    table = pd.DataFrame(data).rename(columns={
        "time": "date",
        "temperature_2m_max": "max_temp",
        "temperature_2m_min": "min_temp",
        "temperature_2m_mean": "avg_temp",
        "precipitation_sum": "rainfall",
        "wind_speed_10m_max": "wind_speed",
    })
    return table, place["name"] + ", " + place.get("country", "")


def get_sample_data(start_year, end_year):
    np.random.seed(42)  # same random numbers every time
    dates = pd.date_range(f"{start_year}-01-01", f"{end_year}-12-31", freq="D")
    day_of_year = dates.dayofyear.to_numpy()

    # temperature goes up in summer and down in winter (a sine wave) + random noise
    avg_temp = 25 + 10 * np.sin(2 * np.pi * (day_of_year - 80) / 365) + np.random.normal(0, 1.8, len(dates))
    max_temp = avg_temp + np.random.uniform(4, 8, len(dates))
    min_temp = avg_temp - np.random.uniform(4, 8, len(dates))

    # more rain in July, August and September
    monsoon = np.isin(dates.month, [7, 8, 9])
    rain_chance = np.where(monsoon, 0.55, 0.10)
    it_rains = np.random.rand(len(dates)) < rain_chance
    rainfall = np.where(it_rains, np.random.gamma(2.0, np.where(monsoon, 9.0, 3.0)), 0.0)
    wind_speed = np.random.normal(18, 6, len(dates)).clip(3, None)

    table = pd.DataFrame({
        "date": dates,
        "max_temp": max_temp.round(1),
        "min_temp": min_temp.round(1),
        "avg_temp": avg_temp.round(1),
        "rainfall": rainfall.round(1),
        "wind_speed": wind_speed.round(1),
    })
    return table


# ---- try real data first, fall back to sample data ----
try:
    if not USE_ONLINE_DATA:
        raise ValueError("Online data switched off")
    df, place_name = get_online_data(CITY, START_YEAR, END_YEAR)
    print("✅ Real weather data downloaded for:", place_name)
except Exception as error:
    print("⚠️ Could not download real data (", error, ")")
    print("➡️ Using sample data instead.")
    df = get_sample_data(START_YEAR, END_YEAR)
    place_name = "Sample City (practice data)"

df["date"] = pd.to_datetime(df["date"])
print("Total days of data:", len(df))

## Step 4: Explore the data
Always look at your data first. Here we check the first rows, the size, the column types and basic statistics.

In [ ]:
df.head()   # first 5 rows

In [ ]:
print("Rows, Columns:", df.shape)
print()
df.info()

In [ ]:
df.describe()   # count, mean, min, max etc. for every column

## Step 5: Clean the data
Real data is rarely perfect. We will:
1. Count missing values
2. Remove duplicate rows
3. Fill small gaps
4. Add new columns (year, month, rainy day) that make analysis easier

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())

In [ ]:
# remove duplicates and sort by date
df = df.drop_duplicates(subset="date").sort_values("date").reset_index(drop=True)

# fill missing temperature / wind values using the neighbouring days
for column in ["max_temp", "min_temp", "avg_temp", "wind_speed"]:
    df[column] = df[column].interpolate(limit_direction="both")

# if rainfall is missing, assume no rain
df["rainfall"] = df["rainfall"].fillna(0)

# new helper columns
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df["rainy_day"] = df["rainfall"] >= 1      # True if 1 mm or more rain

print("Missing values after cleaning:", df.isnull().sum().sum())
df.head()

## Step 6: Analysis with NumPy
NumPy works on **arrays** (lists of numbers) and is very fast. We convert one column into an array and calculate statistics.

In [ ]:
temps = df["avg_temp"].to_numpy()      # Pandas column -> NumPy array
rain = df["rainfall"].to_numpy()

print("Average temperature  :", round(np.mean(temps), 2), "°C")
print("Median temperature   :", round(np.median(temps), 2), "°C")
print("Highest temperature  :", np.max(df["max_temp"].to_numpy()), "°C")
print("Lowest temperature   :", np.min(df["min_temp"].to_numpy()), "°C")
print("Standard deviation   :", round(np.std(temps), 2), "(how much temperature varies)")
print("90th percentile      :", round(np.percentile(temps, 90), 2), "°C (only 10% of days are hotter)")
print()
print("Total rainfall       :", round(np.sum(rain), 1), "mm")
print("Days with rain       :", int(np.sum(rain >= 1)))
print("Days above 35 °C     :", int(np.sum(df["max_temp"].to_numpy() > 35)))

# correlation: does temperature move together with rainfall? (-1 to +1)
correlation = np.corrcoef(temps, rain)[0, 1]
print()
print("Correlation between temperature and rainfall:", round(correlation, 2))

## Step 7: Analysis with Pandas
`groupby()` splits the table into groups (for example by month) and calculates something for each group.

In [ ]:
# Average temperature and total rainfall for each month (across all years)
monthly = df.groupby("month").agg(
    month_name=("month_name", "first"),
    avg_temp=("avg_temp", "mean"),
    total_rain=("rainfall", "sum"),
    rainy_days=("rainy_day", "sum"),
)
number_of_years = df["year"].nunique()
monthly["total_rain"] = monthly["total_rain"] / number_of_years   # average per year
monthly["rainy_days"] = monthly["rainy_days"] / number_of_years
monthly = monthly.round(1)
monthly

In [ ]:
# Yearly summary (only complete years)
yearly = df.groupby("year").agg(
    days=("date", "count"),
    avg_temp=("avg_temp", "mean"),
    total_rain=("rainfall", "sum"),
)
yearly = yearly[yearly["days"] >= 360].round(2)
yearly

In [ ]:
hottest_month = monthly["avg_temp"].idxmax()
coldest_month = monthly["avg_temp"].idxmin()
wettest_month = monthly["total_rain"].idxmax()

print("Hottest month :", monthly.loc[hottest_month, "month_name"])
print("Coldest month :", monthly.loc[coldest_month, "month_name"])
print("Wettest month :", monthly.loc[wettest_month, "month_name"])

In [ ]:
print("🔥 5 hottest days")
print(df.nlargest(5, "max_temp")[["date", "max_temp"]].to_string(index=False))
print()
print("❄️ 5 coldest nights")
print(df.nsmallest(5, "min_temp")[["date", "min_temp"]].to_string(index=False))
print()
print("🌧️ 5 wettest days")
print(df.nlargest(5, "rainfall")[["date", "rainfall"]].to_string(index=False))

## Step 8: Is the temperature rising? (Trend analysis)
`np.polyfit()` draws the best straight line through the yearly averages. The **slope** tells us how much the temperature changes every year.

In [ ]:
years = yearly.index.to_numpy()
yearly_temp = yearly["avg_temp"].to_numpy()

slope, intercept = np.polyfit(years, yearly_temp, 1)   # 1 = straight line

print("Change per year   :", round(slope, 3), "°C")
print("Change per decade :", round(slope * 10, 2), "°C")

if slope > 0:
    print("➡️ The average temperature is RISING over these years.")
else:
    print("➡️ The average temperature is FALLING over these years.")

## Step 9: Visualise the data 📊
Charts make patterns easy to see.

In [ ]:
# Chart 1: daily temperature with a 30-day average
plt.figure(figsize=(12, 5))
plt.plot(df["date"], df["avg_temp"], color="orange", alpha=0.4, label="Daily average")
plt.plot(df["date"], df["avg_temp"].rolling(30).mean(), color="red", linewidth=2, label="30-day average")
plt.title("Daily Temperature - " + place_name)
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Chart 2: average temperature of each month
plt.figure(figsize=(10, 5))
plt.bar(monthly["month_name"], monthly["avg_temp"], color="tomato")
plt.title("Average Temperature by Month - " + place_name)
plt.xlabel("Month")
plt.ylabel("Temperature (°C)")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

In [ ]:
# Chart 3: average rainfall of each month
plt.figure(figsize=(10, 5))
plt.bar(monthly["month_name"], monthly["total_rain"], color="steelblue")
plt.title("Average Monthly Rainfall - " + place_name)
plt.xlabel("Month")
plt.ylabel("Rainfall (mm)")
plt.grid(True, axis="y", alpha=0.3)
plt.show()

In [ ]:
# Chart 4: yearly average temperature with the trend line
plt.figure(figsize=(10, 5))
plt.plot(years, yearly_temp, marker="o", color="red", label="Yearly average")
plt.plot(years, slope * years + intercept, "--", color="black", label="Trend line")
plt.title("Yearly Average Temperature - " + place_name)
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.xticks(years)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Chart 5: how often does each temperature occur? (histogram)
plt.figure(figsize=(10, 5))
plt.hist(df["avg_temp"], bins=25, color="seagreen", edgecolor="white")
plt.title("Distribution of Daily Temperatures - " + place_name)
plt.xlabel("Temperature (°C)")
plt.ylabel("Number of days")
plt.show()

## Step 10: Conclusion
The program prints a short summary of everything we found.

In [ ]:
print("=" * 55)
print("WEATHER ANALYSIS SUMMARY -", place_name)
print("=" * 55)
print("Period analysed       :", df["date"].min().date(), "to", df["date"].max().date())
print("Average temperature   :", round(df["avg_temp"].mean(), 1), "°C")
print("Hottest month         :", monthly.loc[hottest_month, "month_name"])
print("Coldest month         :", monthly.loc[coldest_month, "month_name"])
print("Wettest month         :", monthly.loc[wettest_month, "month_name"])
print("Average yearly rain   :", round(yearly["total_rain"].mean()), "mm")
print("Temperature trend     :", round(slope * 10, 2), "°C per decade")
print("=" * 55)

### Save the cleaned data (optional)
This saves the cleaned table as a CSV file. In Google Colab it will also download to your computer.

In [ ]:
df.to_csv("weather_cleaned_data.csv", index=False)
print("Saved as weather_cleaned_data.csv")

try:
    from google.colab import files
    files.download("weather_cleaned_data.csv")
except ImportError:
    pass   # not running in Colab, so just keep the file

---
### 🚀 Ideas to improve this project
- Compare two cities (for example Delhi and Chennai)
- Predict next month's temperature using machine learning (scikit-learn)
- Build a small web app using Streamlit